In [0]:
#row count check

silver_table = "workspace.superstore.silver_online_superstore" 

current_count = spark.table(silver_table).count()
print(f"Current row count: {current_count}")

MIN_EXPECTED_ROWS = 8000 

if current_count < MIN_EXPECTED_ROWS:
    raise Exception(f"DATA QUALITY FAILURE: Row count dropped to {current_count}, below minimum threshold of {MIN_EXPECTED_ROWS}")

print("✓ Check 1 passed: row count is healthy")

In [0]:
#null check in critical fields

from pyspark.sql.functions import col

null_check = spark.table(silver_table).filter(
    col("product_id").isNull() |
    col("product_name").isNull() |
    col("postal_code").isNull() |
    col("customer_id").isNull() |
    col("transaction_timestamp").isNull() |
    col("receipt_id").isNull()
).count()

print(f"null values in critical fields: {null_check}")

if null_check > 0:
    raise Exception(f"DATA QUALITY FAILURE: Found {null_check} rows with null values in critical fields")

print("✓ Check 2 passed: no nulls in critical fields")

In [0]:
#duplicate rows check

total_rows = spark.table(silver_table).count()

distinct_row_ids = spark.table(silver_table).select("row_id").distinct().count()

print(f"Total rows: {total_rows}, Distinct row_ids: {distinct_row_ids}")

if total_rows != distinct_row_ids:
    duplicate_count = total_rows - distinct_row_ids
    raise Exception(f"DATA QUALITY FAILURE: Found {duplicate_count} duplicate row_id(s)")

print("✓ Check 3 passed: no duplicate rows")